# BME 507 — Warm-up Exercise 6: Instance / Organ Segmentation

**Goal:** predict five anatomical/background classes from `Segmentation02`:

0. outside body  
1. chest area  
2. heart  
3. right lung  
4. left lung  

The original MiniJSRT label uses one lung value for both lungs. This notebook separates the two largest lung connected components; the component appearing on the **left side of the radiograph** is labeled as the patient's **right lung**, consistent with standard chest X-ray display convention.

**Simple experiment:** train the same multiclass U-Net:
1. without augmentation
2. with small rotation/translation + mild brightness augmentation

The notebook saves all metrics, curves, weights, and example masks for the report.

In [ ]:
# ============================================================
# BME 507 - WARM-UP EXERCISE 6
# Instance / organ segmentation
# Experiment: no augmentation vs simple augmentation
# ============================================================

import os
import random
import shutil
import subprocess
import zipfile
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

from google.colab import drive

# ------------------------------------------------------------
# 1. SETTINGS + GOOGLE DRIVE
# ------------------------------------------------------------

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/BME507/Warmup_5_6_7")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = Path("/content/Segmentation02.zip")
EXTRACT_DIR = Path("/content/task6_segmentation02")

DATA_URLS = [
    "https://imgcom.jsrt.or.jp/imgcom/wp-content/uploads/2018/11/Segmentation02.zip",
    "http://imgcom.jsrt.or.jp/imgcom/wp-content/uploads/2018/11/Segmentation02.zip",
]

IMAGE_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 15
LEARNING_RATE = 1e-3
PATIENCE = 5
SEED = 42

CLASS_NAMES = [
    "outside_body",
    "chest_area",
    "heart",
    "right_lung",
    "left_lung",
]
NUM_CLASSES = len(CLASS_NAMES)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. GET + EXTRACT DATASET
# ------------------------------------------------------------

def download_if_needed(zip_path, urls):
    if zip_path.exists() and zip_path.stat().st_size > 1000:
        print("Dataset zip already exists:", zip_path)
        return

    for url in urls:
        print("Trying:", url)
        result = subprocess.run(
            ["wget", "-q", "-O", str(zip_path), url],
            capture_output=True
        )
        if result.returncode == 0 and zip_path.exists() and zip_path.stat().st_size > 1000:
            print("Downloaded:", zip_path)
            return

    raise FileNotFoundError(
        "Could not download Segmentation02.zip automatically. "
        "Upload Segmentation02.zip manually to /content/Segmentation02.zip and rerun."
    )

def find_named_dir(root, name):
    for current, dirs, files in os.walk(root):
        if Path(current).name.lower() == name.lower():
            return Path(current)
    raise FileNotFoundError(f"Folder '{name}' was not found under {root}")

def pair_images_and_masks(image_dir, mask_dir):
    exts = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

    images = {
        p.stem: p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    }
    masks = {
        p.stem: p for p in mask_dir.iterdir()
        if p.is_file() and p.suffix.lower() in exts
    }

    common = sorted(set(images) & set(masks))
    if not common:
        raise RuntimeError(
            f"No matching image/mask filenames found in:\n{image_dir}\n{mask_dir}"
        )

    return [(images[k], masks[k]) for k in common]

download_if_needed(ZIP_PATH, DATA_URLS)

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(EXTRACT_DIR)

org_train = find_named_dir(EXTRACT_DIR, "org_train")
label_train = find_named_dir(EXTRACT_DIR, "label_train")
org_test = find_named_dir(EXTRACT_DIR, "org_test")
label_test = find_named_dir(EXTRACT_DIR, "label_test")

all_train_pairs = pair_images_and_masks(org_train, label_train)
test_pairs = pair_images_and_masks(org_test, label_test)

print("Training pairs:", len(all_train_pairs))
print("Official test pairs:", len(test_pairs))

# Inspect raw label values
sample_raw = np.asarray(Image.open(all_train_pairs[0][1]).convert("L"))
print("Example raw mask values:", np.unique(sample_raw))

# Reproducible train/validation split
indices = np.arange(len(all_train_pairs))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n_val = max(1, int(round(0.20 * len(indices))))
val_indices = indices[:n_val]
train_indices = indices[n_val:]

train_pairs = [all_train_pairs[i] for i in train_indices]
val_pairs = [all_train_pairs[i] for i in val_indices]

print("Train:", len(train_pairs), "| Validation:", len(val_pairs), "| Test:", len(test_pairs))

# ------------------------------------------------------------
# 3. MASK CONVERSION
# ------------------------------------------------------------

def split_lung_components(lung_binary):
    """
    Return two binary masks:
    right lung = patient's right = image-left component
    left lung  = patient's left  = image-right component
    """
    lung_u8 = lung_binary.astype(np.uint8)

    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        lung_u8, connectivity=8
    )

    component_ids = list(range(1, n_labels))
    component_ids.sort(
        key=lambda k: stats[k, cv2.CC_STAT_AREA],
        reverse=True
    )
    component_ids = component_ids[:2]

    h, w = lung_u8.shape

    if len(component_ids) < 2:
        # Safe fallback if connected components do not return two lungs.
        x = np.arange(w)[None, :]
        right = lung_u8.astype(bool) & (x < w // 2)
        left = lung_u8.astype(bool) & (x >= w // 2)
        return right, left

    a, b = component_ids
    if centroids[a][0] < centroids[b][0]:
        right_id, left_id = a, b
    else:
        right_id, left_id = b, a

    right = labels == right_id
    left = labels == left_id

    return right, left

def raw_mask_to_classes(raw):
    """
    MiniJSRT Segmentation02 values used here:
      0   -> outside body
      170 -> chest area / outside lungs
      85  -> heart
      255 -> lung region (split into right/left)
    """
    class_mask = np.zeros_like(raw, dtype=np.int64)

    class_mask[raw == 170] = 1
    class_mask[raw == 85] = 2

    lung = raw == 255
    right_lung, left_lung = split_lung_components(lung)

    class_mask[right_lung] = 3
    class_mask[left_lung] = 4

    return class_mask

# ------------------------------------------------------------
# 4. DATASET + AUGMENTATION
# ------------------------------------------------------------

class OrganDataset(Dataset):
    def __init__(self, pairs, augment=False):
        self.pairs = pairs
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        image_path, mask_path = self.pairs[idx]

        image = Image.open(image_path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR
        )
        mask = Image.open(mask_path).convert("L").resize(
            (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.NEAREST
        )

        if self.augment:
            angle = random.uniform(-5.0, 5.0)
            max_shift = int(0.03 * IMAGE_SIZE)
            translate = (
                random.randint(-max_shift, max_shift),
                random.randint(-max_shift, max_shift),
            )

            image = TF.affine(
                image,
                angle=angle,
                translate=translate,
                scale=1.0,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.BILINEAR,
                fill=0,
            )

            mask = TF.affine(
                mask,
                angle=angle,
                translate=translate,
                scale=1.0,
                shear=[0.0, 0.0],
                interpolation=InterpolationMode.NEAREST,
                fill=0,
            )

            brightness = random.uniform(0.90, 1.10)
            image = ImageEnhance.Brightness(image).enhance(brightness)

        image_np = np.asarray(image, dtype=np.float32) / 255.0
        raw_mask = np.asarray(mask, dtype=np.uint8)
        class_mask = raw_mask_to_classes(raw_mask)

        image_t = torch.from_numpy(image_np).unsqueeze(0)
        mask_t = torch.from_numpy(class_mask).long()

        return image_t, mask_t, image_path.name

def make_loaders(augment_training):
    train_dataset = OrganDataset(train_pairs, augment=augment_training)
    val_dataset = OrganDataset(val_pairs, augment=False)

    return (
        DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=2,
            pin_memory=(device.type == "cuda"),
        ),
        DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=2,
            pin_memory=(device.type == "cuda"),
        ),
    )

test_dataset = OrganDataset(test_pairs, augment=False)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
)

# ------------------------------------------------------------
# 5. SMALL MULTICLASS U-NET
# ------------------------------------------------------------

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=5, base=16):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)

        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = DoubleConv(base * 16, base * 8)

        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)

        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)

        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)

        self.out = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))

        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))

        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out(d1)

criterion = nn.CrossEntropyLoss()

# ------------------------------------------------------------
# 6. MULTICLASS METRICS
# ------------------------------------------------------------

def multiclass_metrics(pred, target, num_classes=NUM_CLASSES, eps=1e-7):
    dices = []
    ious = []

    for class_id in range(num_classes):
        pred_c = pred == class_id
        target_c = target == class_id

        intersection = (pred_c & target_c).sum().item()
        pred_count = pred_c.sum().item()
        target_count = target_c.sum().item()
        union = pred_count + target_count - intersection

        dice = (2 * intersection + eps) / (pred_count + target_count + eps)
        iou = (intersection + eps) / (union + eps)

        dices.append(dice)
        ious.append(iou)

    return np.array(dices), np.array(ious)

def evaluate_loader(model, loader):
    model.eval()

    losses = []
    dice_values = []
    iou_values = []
    pixel_accuracies = []

    with torch.no_grad():
        for images, masks, _ in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)
            pred = logits.argmax(dim=1)

            losses.append(loss.item())
            pixel_accuracies.append(
                (pred == masks).float().mean().item()
            )

            for b in range(images.size(0)):
                dice, iou = multiclass_metrics(
                    pred[b].cpu(),
                    masks[b].cpu()
                )
                dice_values.append(dice)
                iou_values.append(iou)

    dice_values = np.stack(dice_values)
    iou_values = np.stack(iou_values)

    return {
        "loss": float(np.mean(losses)),
        "mean_dice": float(dice_values.mean()),
        "mean_iou": float(iou_values.mean()),
        "pixel_accuracy": float(np.mean(pixel_accuracies)),
        "dice_per_class": dice_values.mean(axis=0),
        "iou_per_class": iou_values.mean(axis=0),
    }

# ------------------------------------------------------------
# 7. TRAINING
# ------------------------------------------------------------

def train_experiment(experiment_name, augment_training):
    train_loader, val_loader = make_loaders(augment_training)

    model = UNet(in_channels=1, out_channels=NUM_CLASSES, base=16).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    history = []
    best_val_dice = -1.0
    best_state = None
    patience_counter = 0

    print("\n" + "=" * 70)
    print("TRAINING:", experiment_name)
    print("=" * 70)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        seen = 0

        for images, masks, _ in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            seen += images.size(0)

        train_loss = running_loss / seen
        val = evaluate_loader(model, val_loader)

        history.append({
            "experiment": experiment_name,
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val["loss"],
            "val_mean_dice": val["mean_dice"],
            "val_mean_iou": val["mean_iou"],
            "val_pixel_accuracy": val["pixel_accuracy"],
        })

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} | "
            f"val_mean_dice={val['mean_dice']:.4f} | "
            f"val_mean_iou={val['mean_iou']:.4f}"
        )

        if val["mean_dice"] > best_val_dice:
            best_val_dice = val["mean_dice"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)

    weight_path = SAVE_DIR / f"task6_{experiment_name}_best.pth"
    torch.save(model.state_dict(), weight_path)

    print("Saved model:", weight_path)
    return model, pd.DataFrame(history)

# ------------------------------------------------------------
# 8. RUN BOTH EXPERIMENTS
# ------------------------------------------------------------

experiment_specs = [
    ("no_augmentation", False),
    ("augmented", True),
]

models = {}
histories = []
summary_rows = []
per_class_rows = []

for experiment_name, augment_training in experiment_specs:
    model, history_df = train_experiment(
        experiment_name,
        augment_training
    )

    models[experiment_name] = model
    histories.append(history_df)

    test = evaluate_loader(model, test_loader)

    summary_rows.append({
        "experiment": experiment_name,
        "test_mean_dice": test["mean_dice"],
        "test_mean_iou": test["mean_iou"],
        "test_pixel_accuracy": test["pixel_accuracy"],
        "best_val_mean_dice": float(history_df["val_mean_dice"].max()),
        "epochs_completed": int(history_df["epoch"].max()),
    })

    for class_id, class_name in enumerate(CLASS_NAMES):
        per_class_rows.append({
            "experiment": experiment_name,
            "class_id": class_id,
            "class_name": class_name,
            "test_dice": float(test["dice_per_class"][class_id]),
            "test_iou": float(test["iou_per_class"][class_id]),
        })

history_all = pd.concat(histories, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)
per_class_df = pd.DataFrame(per_class_rows)

history_all.to_csv(SAVE_DIR / "task6_training_history.csv", index=False)
summary_df.to_csv(SAVE_DIR / "task6_summary.csv", index=False)
per_class_df.to_csv(SAVE_DIR / "task6_per_class_metrics.csv", index=False)

print("\nFINAL TASK 6 RESULTS")
print(summary_df)
print("\nPER-CLASS RESULTS")
print(per_class_df)

# ------------------------------------------------------------
# 9. SELECT BEST MODEL USING VALIDATION DICE
# ------------------------------------------------------------

best_experiment = summary_df.sort_values(
    "best_val_mean_dice",
    ascending=False
).iloc[0]["experiment"]

best_model = models[best_experiment]

torch.save(
    best_model.state_dict(),
    SAVE_DIR / "task6_best_model.pth"
)

(SAVE_DIR / "task6_best_model_name.txt").write_text(str(best_experiment))

print("\nBest model selected from validation Dice:", best_experiment)

# ------------------------------------------------------------
# 10. SAVE TRAINING CURVES
# ------------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name in summary_df["experiment"]:
    d = history_all[history_all["experiment"] == name]
    plt.plot(d["epoch"], d["train_loss"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Task 6 - Training loss")
plt.legend()

plt.subplot(1, 2, 2)
for name in summary_df["experiment"]:
    d = history_all[history_all["experiment"] == name]
    plt.plot(d["epoch"], d["val_mean_dice"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation mean Dice")
plt.title("Task 6 - Validation mean Dice")
plt.legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / "task6_training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 11. SAVE QUALITATIVE EXAMPLES
# ------------------------------------------------------------

n_examples = min(4, len(test_dataset))
example_indices = np.linspace(0, len(test_dataset) - 1, n_examples, dtype=int)

plt.figure(figsize=(12, 3 * n_examples))

for row, idx in enumerate(example_indices):
    image, mask, filename = test_dataset[idx]

    predictions = {}
    for name in ["no_augmentation", "augmented"]:
        models[name].eval()
        with torch.no_grad():
            logits = models[name](image.unsqueeze(0).to(device))
            predictions[name] = logits.argmax(dim=1).cpu().squeeze().numpy()

    panels = [
        (image.squeeze().numpy(), f"X-ray\n{filename}", "gray", None, None),
        (mask.numpy(), "Ground truth", "tab10", 0, NUM_CLASSES - 1),
        (predictions["no_augmentation"], "No augmentation", "tab10", 0, NUM_CLASSES - 1),
        (predictions["augmented"], "Augmented", "tab10", 0, NUM_CLASSES - 1),
    ]

    for col, (arr, title, cmap, vmin, vmax) in enumerate(panels):
        ax = plt.subplot(n_examples, 4, row * 4 + col + 1)
        ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.axis("off")

plt.tight_layout()
plt.savefig(SAVE_DIR / "task6_examples.png", dpi=300, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# 12. SAVE RUN INFORMATION
# ------------------------------------------------------------

run_info = [
    f"device={device}",
    f"gpu={torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}",
    f"image_size={IMAGE_SIZE}",
    f"batch_size={BATCH_SIZE}",
    f"max_epochs={EPOCHS}",
    f"learning_rate={LEARNING_RATE}",
    f"train_images={len(train_pairs)}",
    f"validation_images={len(val_pairs)}",
    f"test_images={len(test_pairs)}",
    f"best_experiment={best_experiment}",
    "class_names=" + ",".join(CLASS_NAMES),
]

(SAVE_DIR / "task6_run_info.txt").write_text("\n".join(run_info))

print("\nSaved Task 6 outputs to:", SAVE_DIR)